In [7]:
import pandas as pd
from utils.notebook_helpers import get_postgres_engine

engine, pg_conf = get_postgres_engine("../configs/database.yaml")
print("Connected to PostgreSQL:", pg_conf["host"], pg_conf["db"])

Connected to PostgreSQL: localhost storage_analytics


In [8]:
anomaly_df = pd.read_sql_query("SELECT * FROM anomaly_events", engine)
curated_df = pd.read_sql_query("SELECT * FROM curated_device_metrics", engine)

curated_df["timestamp"] = pd.to_datetime(curated_df["timestamp"])
anomaly_df["timestamp"] = pd.to_datetime(anomaly_df["timestamp"])

anomaly_df.head()

,device,timestamp,metric_name,metric_value,detector_type,anomaly_score,severity,is_anomaly,details,source_file,...,util_pct,aqu_sz,avg_latency_ms,read_ratio,write_ratio,avg_request_size_kb,total_iops,total_throughput_mb_s,saturation_score,latency_pressure
0,dm-0,2026-03-31 07:28:52.783839+00:00,avg_latency_ms,2.035165,iqr,0.339752,medium,1,"{""mean"": 1.2965746994857374, ""std"": 2.17391134...",data/raw/generated_iostat.csv,...,47.512542,1.207108,2.035165,0.528114,0.471886,73.283216,400.057592,28.630378,57.352765,2.456664
1,dm-0,2026-03-31 08:24:23.158227+00:00,avg_latency_ms,1.877753,iqr,0.267342,medium,1,"{""mean"": 1.2965746994857374, ""std"": 2.17391134...",data/raw/generated_iostat.csv,...,31.819139,2.141826,1.877753,0.523356,0.476644,56.526004,491.066143,27.107429,68.151073,4.021820
2,dm-0,2026-03-31 10:49:01.187180+00:00,avg_latency_ms,1.834332,iqr,0.247368,medium,1,"{""mean"": 1.2965746994857374, ""std"": 2.17391134...",data/raw/generated_iostat.csv,...,40.728417,4.678540,1.834332,0.531697,0.468303,56.940489,773.635080,43.018711,190.549511,8.581993
3,dm-0,2026-03-31 13:59:32.617324+00:00,avg_latency_ms,1.858058,iqr,0.258283,medium,1,"{""mean"": 1.2965746994857374, ""std"": 2.17391134...",data/raw/generated_iostat.csv,...,43.368571,0.814004,1.858058,0.478716,0.521284,84.052996,694.166454,56.979268,35.302170,1.512466
4,dm-0,2026-03-31 16:44:15.747940+00:00,avg_latency_ms,1.951055,iqr,0.301061,medium,1,"{""mean"": 1.2965746994857374, ""std"": 2.17391134...",data/raw/generated_iostat.csv,...,51.895259,1.210086,1.951055,0.408910,0.591090,84.859282,566.691984,46.961987,62.797744,2.360945


In [9]:
anomaly_df["detector_type"].value_counts()
anomaly_df["severity"].value_counts()
anomaly_df["metric_name"].value_counts()

metric_name
saturation_score         2264
iowait_pressure          1843
avg_latency_ms           1520
queue_efficiency         1320
total_iops                672
merge_efficiency          551
total_throughput_mb_s     253
Name: count, dtype: int64

In [10]:
anomaly_df.groupby("device").size().sort_values(ascending=False)

device
sdb        2526
sda        1604
nvme1n1    1520
nvme0n1    1391
dm-0       1382
dtype: int64

In [11]:
context_cols = [
    "device", "timestamp", "merge_efficiency", "queue_efficiency",
    "iowait_pressure", "svctm_await_ratio", "write_amplification",
    "workload_pattern", "saturation_score",
 ]
joined = anomaly_df.merge(curated_df[context_cols], on=["device", "timestamp"], how="left")
joined.sort_values("anomaly_score", ascending=False).head(20)

,device,timestamp,metric_name,metric_value,detector_type,anomaly_score,severity,is_anomaly,details,source_file,...,total_throughput_mb_s,saturation_score_x,latency_pressure,merge_efficiency,queue_efficiency,iowait_pressure,svctm_await_ratio,write_amplification,workload_pattern_y,saturation_score_y
7556,sdb,2026-04-03 13:59:12.206839+00:00,avg_latency_ms,453008.805530,zscore+iqr,44.796539,critical,1,"{""mean"": 426.53597923945534, ""std"": 10103.0633...",data/raw/generated_iostat.csv,...,73.438128,333.631862,1.511382e+06,0.251485,326.895604,81.103507,0.000043,3.683423,latency_sensitive,333.631862
2058,sdb,2026-03-31 10:49:09.985928+00:00,avg_latency_ms,37844.649680,zscore+iqr,39.676975,critical,1,"{""mean"": 105.59038058176263, ""std"": 951.157680...",data/raw/generated_iostat.csv,...,19.674806,118.888941,9.541118e+04,0.336629,85.254732,5.414408,0.000027,1.290928,latency_sensitive,118.888941
4791,sdb,2026-04-01 18:17:11.957126+00:00,avg_latency_ms,97465.412165,zscore+iqr,38.657081,critical,1,"{""mean"": 163.69124146957063, ""std"": 2517.04779...",data/raw/generated_iostat.csv,...,10.338320,82.984604,1.485967e+05,0.389143,61.069963,3.917507,0.000047,1.422752,latency_sensitive,82.984604
2831,dm-0,2026-04-03 08:01:35.058329+00:00,avg_latency_ms,24.715446,zscore+iqr,25.997800,critical,1,"{""mean"": 1.088108558650823, ""std"": 0.908820645...",data/raw/generated_iostat.csv,...,30.204842,93.592979,7.099155e+01,0.071955,136.502495,4.671664,0.251485,0.904358,latency_sensitive,93.592979
7035,sda,2026-04-06 16:24:13.900400+00:00,avg_latency_ms,54.376878,zscore+iqr,25.457589,critical,1,"{""mean"": 1.6331583169935282, ""std"": 2.07182700...",data/raw/generated_iostat.csv,...,52.976914,18.162339,5.334345e+01,0.049210,1202.548272,4.796046,0.010716,6.320710,latency_sensitive,18.162339
2832,dm-0,2026-04-03 08:06:45.573564+00:00,avg_latency_ms,23.947422,zscore+iqr,25.152723,critical,1,"{""mean"": 1.088108558650823, ""std"": 0.908820645...",data/raw/generated_iostat.csv,...,40.359936,187.800535,1.208113e+02,0.063584,99.605591,7.429945,0.241018,0.947898,latency_sensitive,187.800535
423,dm-0,2026-04-01 08:18:34.624194+00:00,iowait_pressure,47.576702,zscore+iqr,23.859481,critical,1,"{""mean"": 0.9213049573797445, ""std"": 1.95542380...",data/raw/generated_iostat.csv,...,36.253131,488.318627,4.129233e+01,0.086652,96.997767,47.576702,0.185090,0.750613,latency_sensitive,488.318627
122,dm-0,2026-04-01 07:54:10.512457+00:00,saturation_score,1873.960508,zscore+iqr,21.884856,critical,1,"{""mean"": 37.68711545152062, ""std"": 83.90612245...",data/raw/generated_iostat.csv,...,23.699673,1873.960508,1.494619e+02,0.080706,15.931898,25.275417,0.202720,0.880933,latency_sensitive,1873.960508
5054,sdb,2026-04-01 13:26:58.378802+00:00,saturation_score,3871.579221,zscore+iqr,21.648131,critical,1,"{""mean"": 74.96597194720587, ""std"": 175.3783414...",data/raw/generated_iostat.csv,...,21.393932,3871.579221,3.213458e+02,0.331904,5.243671,31.459683,0.557883,1.205945,latency_sensitive,3871.579221
2645,sdb,2026-04-05 23:38:39.349708+00:00,queue_efficiency,2991.886142,zscore+iqr,21.044275,critical,1,"{""mean"": 116.62037940780895, ""std"": 136.629360...",data/raw/generated_iostat.csv,...,7.711844,0.332543,1.548784e+00,0.210373,2991.886142,1.047156,0.126965,8.095226,latency_sensitive,0.332543


In [12]:
focus_metrics = ["iowait_pressure", "merge_efficiency", "queue_efficiency", "avg_latency_ms", "saturation_score"]
metric_workload = anomaly_df.groupby(["metric_name", "workload_pattern"]).size().sort_values(ascending=False).head(20)
focus_severity = (
    anomaly_df[anomaly_df["metric_name"].isin(focus_metrics)]
    .groupby(["metric_name", "severity"]).size()
    .unstack(fill_value=0)
    .sort_index()
)
{"metric_workload_top20": metric_workload, "focus_metric_severity": focus_severity}

{'metric_workload_top20': metric_name            workload_pattern 
 saturation_score       balanced             1091
 avg_latency_ms         latency_sensitive    1079
 iowait_pressure        balanced              877
 queue_efficiency       balanced              721
 saturation_score       latency_sensitive     491
 merge_efficiency       balanced              460
 queue_efficiency       latency_sensitive     403
 iowait_pressure        burst_io              388
 saturation_score       burst_io              385
 avg_latency_ms         balanced              360
 iowait_pressure        latency_sensitive     333
 saturation_score       saturated             276
 total_iops             latency_sensitive     268
                        burst_io              211
 iowait_pressure        saturated             189
 total_throughput_mb_s  latency_sensitive     125
 queue_efficiency       burst_io               97
 total_throughput_mb_s  balanced               90
 merge_efficiency       latency_s